In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"]="0"
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
np.random.seed(18)
from scipy.ndimage import gaussian_filter
import tensorflow as tf

from func_file_Model import ResNet_model_paper
from func_file_Data import Generate_normed_dataset

## Load the DAMN model

Tensorflow may print warnings despite operating correctly.

In [ ]:
#Load the DAMN model (ResNet architecture)
model_ResNet = ResNet_model_paper(channels = 64, num_blocks_array = [3,3,3,3], 
                                  kernel_sizes=[5,7,9,11], LeakyReLU_slope=0.1, 
                                  padding="same", interpolation="bilinear", kernel_initializer="he_uniform")

#Compile and load weights
_ = model_ResNet(tf.random.normal([1, 50, 50, 1]))
model_ResNet.load_weights("DAMN_ResNet.keras")

In [ ]:
#Visualize the model summary
model_ResNet.summary()

## Load pre-generated sample to verify correct operation

The pre-generated sample is already correctly normalized.<br>
Tensorflow may print warnings despite operating correctly.

In [ ]:
#Load and extract the test sample
datafile = np.load("Pre_generated_test_sample.npz")

low_res_image = datafile["low_res_image"]
high_res_image = datafile["high_res_image"]
expected_reconstruction = datafile["expected_reconstruction"]

#Use the DAMN model on the low resolution image
reconstructed_sample = np.squeeze(model_ResNet.predict(np.expand_dims(low_res_image, axis=(0,-1))))

#Note: Tensorflow expects data of shape [number_of_sample, height, width, channels]
#Since we use a single intensity image, number_of_sample = 1 = channels

In [ ]:
#Check the reconstruction
print("Expected shape of the reconstruction: (400, 400)")
print("Obtained shape of the reconstruction:", reconstructed_sample.shape, "\n")

print("Does the obtained recontruction match the expected?", np.all(expected_reconstruction == reconstructed_sample))

In [ ]:
#Vizualize the test sample
#Note that vizualizing 400x400 images can hide the emitting pixels
#Convolving the large images with a small Gaussian filter for better vizualization

plt.figure(figsize=(15,5))
plt.subplot(131)
plt.matshow(low_res_image, fignum=False)
plt.title("Low resolution")

plt.subplot(132)
plt.matshow(gaussian_filter(high_res_image, 2), fignum=False)
plt.title("High resolution")

plt.subplot(133)
plt.matshow(gaussian_filter(reconstructed_sample, 2), fignum=False)
plt.title("Reconstruction")

plt.show()

In [ ]:
#Vizualize the test sample close-up without Gaussian filter

plt.figure(figsize=(10,5))
plt.subplot(121)
plt.matshow(high_res_image[50:130,60:150], fignum=False)
plt.title("High resolution - close-up")

plt.subplot(122)
plt.matshow(reconstructed_sample[50:130,60:150], fignum=False)
plt.title("Reconstruction - close-up")

plt.show()

## Generate custom data to demonstrate operation

The generated data are already correctly normalized.

In [ ]:
#Set the parameters for data generation
num_data = 100                   #Number of image pairs to generate
size = 50                        #Size of the low-res images
upsampling_factor = 8            #Upsampling for the ground truth
emit_power = 5000                #Average emitted intensity by each emitter
noise_inten = 10                 #Average shot noise
PSF_width = 2                    #Point spread function width
kernel_choice = 0                #Point spread function type: Gaussian = 0, Airy = 1
concentration = 50               #Number of emitters in each sample

In [ ]:
#Generate the dataset
low_res, high_res = Generate_normed_dataset(num_data=num_data, size=size, upsampling_factor=upsampling_factor, emit_power=emit_power, 
                                            noise_inten=noise_inten, PSF_width=PSF_width, kernel_choice=kernel_choice, concentration=concentration)

In [ ]:
#Use the DAMN model on the low resolution images
#Note: takes about 5 seconds on our computational resources + initialization time (up to a minute the first run time)
reconstructed = np.squeeze(model_ResNet.predict(np.expand_dims(low_res, axis=-1)))

print("Reconstructed images of shape:", reconstructed.shape)

### Progress with any desired evaluation:

In [ ]:
#Your code goes here